# Menangani Data Tidak Seimbang (Imbalanced Data)

Course 3: Supervised Learning, Klasifikasi

Pada banyak masalah klasifikasi, satu kelas jauh lebih sedikit dari kelas lainnya, padahal kelas yang sedikit itulah yang paling penting (penipuan, pelanggan yang berhenti, penyakit). Di notebook ini aku mempelajari kenapa akurasi menyesatkan di data seperti itu, lalu mencoba tiga cara penanganan:

- **class weight**: kelas minoritas diberi bobot lebih besar saat training
- **oversampling (SMOTE)**: membuat data sintetis untuk kelas minoritas
- **undersampling**: membuang sebagian data kelas mayoritas

Empat skenario dipakai: deteksi penipuan kartu kredit (sangat timpang), prediksi pelanggan berhenti (churn), estimasi jenis tumor (agak timpang), dan prediksi pergantian pekerjaan (latihan).

## Dataset

Empat file CSV, semuanya dibaca langsung dari link (kolom target bernama `Class`):

- `im_credit.csv`: transaksi kartu kredit, fitur sudah hasil PCA
- `im_churn.csv`: pelanggan perusahaan telekomunikasi
- `im_cancer.csv`: sampel tumor
- `im_hr.csv`: karyawan, target apakah sedang mencari pekerjaan baru

Alamat dasarnya: `https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-ML201EN-SkillsNetwork/labs/module_4/datasets/`

## 1. Import

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import (accuracy_score, precision_score, recall_score, fbeta_score,
                             roc_auc_score, confusion_matrix)

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
RS = 123
BASE = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-ML201EN-SkillsNetwork/labs/module_4/datasets/"

## 2. Fungsi bantu

Kalau `RandomUnderSampler` dan `SMOTE` dari library `imbalanced-learn` tidak ingin dipasang, keduanya mudah dibuat sendiri, jadi di sini aku tulis versi sederhananya (khusus kasus biner):

- **Undersampling**: dari tiap kelas diambil sampel acak sebanyak jumlah kelas minoritas.
- **SMOTE**: untuk tiap data sintetis, pilih satu data minoritas secara acak, pilih satu dari `k` tetangga terdekatnya (sesama minoritas), lalu ambil titik acak di garis lurus di antara keduanya.

In [ ]:
def pisah_data(df, target="Class"):
    X = df.drop(columns=target)
    y = df[target].astype(int)
    return train_test_split(X, y, test_size=0.2, stratify=y, random_state=RS)


def undersample(X, y, seed=RS):
    rng = np.random.default_rng(seed)
    n_min = y.value_counts().min()
    idx = np.concatenate([rng.choice(np.where(y.values == k)[0], n_min, replace=False)
                          for k in y.unique()])
    return X.iloc[idx], y.iloc[idx]


def smote(X, y, k=5, seed=RS):
    rng = np.random.default_rng(seed)
    minoritas = y.value_counts().idxmin()
    n_butuh = y.value_counts().max() - y.value_counts().min()
    Xm = X[y.values == minoritas].to_numpy(dtype=float)

    nn = NearestNeighbors(n_neighbors=min(k + 1, len(Xm))).fit(Xm)
    tetangga = nn.kneighbors(Xm, return_distance=False)[:, 1:]   # buang dirinya sendiri

    dasar = rng.integers(0, len(Xm), n_butuh)
    pilih = tetangga[dasar, rng.integers(0, tetangga.shape[1], n_butuh)]
    sintetis = Xm[dasar] + rng.random((n_butuh, 1)) * (Xm[pilih] - Xm[dasar])

    X_baru = pd.concat([X, pd.DataFrame(sintetis, columns=X.columns)], ignore_index=True)
    y_baru = pd.concat([y, pd.Series(np.full(n_butuh, minoritas))], ignore_index=True)
    return X_baru, y_baru

In [ ]:
BETA = 2   # F-beta dengan beta > 1: recall dianggap lebih penting daripada precision

def evaluasi(model, X_test, y_test, nama):
    pred = model.predict(X_test)
    skor = model.predict_proba(X_test)[:, 1]
    return {
        "model": nama,
        "accuracy": accuracy_score(y_test, pred),
        "precision": precision_score(y_test, pred, zero_division=0),
        "recall": recall_score(y_test, pred, zero_division=0),
        f"F{BETA}": fbeta_score(y_test, pred, beta=BETA, zero_division=0),
        "AUC": roc_auc_score(y_test, skor),
    }


def bandingkan(df, buat_model, bobot):
    # melatih 4 versi: asli, class weight, SMOTE, undersampling
    X_train, X_test, y_train, y_test = pisah_data(df)
    X_smo, y_smo = smote(X_train, y_train)
    X_und, y_und = undersample(X_train, y_train)

    varian = [
        ("Asli", buat_model(None), X_train, y_train),
        ("Class weight", buat_model(bobot), X_train, y_train),
        ("SMOTE", buat_model(None), X_smo, y_smo),
        ("Undersampling", buat_model(None), X_und, y_und),
    ]
    baris = []
    for nama, model, Xa, ya in varian:
        model.fit(Xa, ya)
        baris.append(evaluasi(model, X_test, y_test, nama))
    return pd.DataFrame(baris).set_index("model")


def grafik(hasil, judul):
    ax = hasil.T.plot.bar(figsize=(10, 4), width=0.8)
    ax.set_ylabel("skor")
    ax.set_title(judul)
    plt.xticks(rotation=0)
    plt.legend(loc="lower left")
    plt.show()


def model_lr(bobot):
    return LogisticRegression(max_iter=1000, class_weight=bobot, random_state=RS)

def model_rf(bobot):
    return RandomForestClassifier(n_estimators=50, max_depth=10, min_samples_split=5,
                                  class_weight=bobot, random_state=RS)

## 3. Skenario 1: deteksi penipuan kartu kredit

In [ ]:
credit = pd.read_csv(BASE + "im_credit.csv", index_col=False)
print(credit.shape)
print(credit["Class"].value_counts().to_string())
print("rasio negatif : positif = ", round(credit["Class"].value_counts()[0] / credit["Class"].value_counts()[1]), ": 1")
credit["Class"].value_counts().plot.bar(color=["tab:green", "tab:red"], figsize=(5, 3))
plt.show()

Penipuan hanya sebagian sangat kecil dari seluruh transaksi, sampai batang kelas positif nyaris tidak terlihat.

### Kenapa akurasi menyesatkan

In [ ]:
X_train, X_test, y_train, y_test = pisah_data(credit)
lr_asli = model_lr(None).fit(X_train, y_train)

pred = lr_asli.predict(X_test)
print(f"akurasi: {accuracy_score(y_test, pred):.4f}")
print(f"kalau model menebak 'bukan penipuan' untuk semua transaksi, akurasi = {1 - y_test.mean():.4f}")
print()
tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()
print(f"penipuan terdeteksi: {tp} | penipuan terlewat: {fn} | transaksi normal salah ditandai: {fp}")

Akurasinya sangat tinggi, tetapi hampir setara dengan model yang menebak "normal" untuk semuanya. Yang penting untuk kasus ini adalah berapa penipuan yang berhasil ditemukan, yaitu recall. Karena itu aku pakai precision, recall, F-beta (beta = 2, recall lebih diutamakan), dan AUC.

### Membandingkan empat cara

In [ ]:
hasil_credit = bandingkan(credit, model_lr, bobot={0: 0.1, 1: 0.9})
hasil_credit.round(4)

In [ ]:
grafik(hasil_credit, "Deteksi penipuan: logistic regression")

Cara membacanya: SMOTE dan undersampling mengubah distribusi data latih menjadi 50:50 padahal data aslinya sangat timpang, sehingga model cenderung terlalu sering menandai transaksi sebagai penipuan (recall naik, precision turun drastis). Class weight bekerja di fungsi loss tanpa mengubah datanya, jadi hasilnya sering lebih seimbang. Untuk data yang sangat timpang, oversampling umumnya kurang cocok karena menggeser distribusi data asli terlalu jauh.

## 4. Skenario 2: prediksi pelanggan berhenti (churn)

In [ ]:
churn = pd.read_csv(BASE + "im_churn.csv", index_col=False)
print(churn.shape)
print((churn["Class"].value_counts(normalize=True) * 100).round(1).to_string())

Pelanggan yang tidak berhenti sekitar empat kali lipat yang berhenti. Dataset ini punya banyak fitur, jadi aku pakai random forest dengan parameter yang sudah dituning dari lab asli (kedalaman 10, 50 pohon, `min_samples_split` 5). Bobot kelas 0.2 untuk tidak berhenti dan 0.8 untuk berhenti.

In [ ]:
hasil_churn = bandingkan(churn, model_rf, bobot={0: 0.2, 1: 0.8})
hasil_churn.round(4)

In [ ]:
grafik(hasil_churn, "Prediksi churn: random forest")

Bandingkan baris `Asli` dengan tiga baris lainnya. Biasanya recall dan AUC naik setelah penanganan, sementara precision turun. Untuk churn, menurunkan precision sering bisa diterima, karena biaya menghubungi pelanggan yang sebenarnya tidak akan pergi lebih kecil daripada kehilangan pelanggan.

## 5. Skenario 3: estimasi jenis tumor

In [ ]:
cancer = pd.read_csv(BASE + "im_cancer.csv", index_col=False)
print(cancer.shape)
print((cancer["Class"].value_counts(normalize=True) * 100).round(1).to_string())

In [ ]:
hasil_cancer = bandingkan(cancer, model_rf, bobot={0: 0.2, 1: 0.8})
hasil_cancer.round(4)

In [ ]:
grafik(hasil_cancer, "Estimasi jenis tumor: random forest")

Dataset ini tidak terlalu timpang dan polanya jelas, jadi model asli biasanya sudah bagus dan penanganan ketidakseimbangan tidak banyak membantu. Tidak semua data tidak seimbang perlu perlakuan khusus.

## 6. Latihan: prediksi pergantian pekerjaan

Data HR berisi profil karyawan, dan targetnya apakah ia sedang mencari pekerjaan baru. Langkahnya: cek distribusi kelas, pisahkan data, lalu bandingkan logistic regression dengan empat cara di atas.

In [ ]:
hr = pd.read_csv(BASE + "im_hr.csv", index_col=False)
print(hr.shape)
print((hr["Class"].value_counts(normalize=True) * 100).round(1).to_string())
hr["Class"].value_counts().plot.bar(color=["tab:green", "tab:red"], figsize=(5, 3))
plt.show()

Bobot kelas yang dipakai tadi (0.1 dan 0.9) berasal dari lab asli. Bobot yang bagus juga bisa dicari dengan grid search pada data train, dengan F1 sebagai ukuran:

In [ ]:
X_train, X_test, y_train, y_test = pisah_data(hr)

grid = GridSearchCV(
    LogisticRegression(max_iter=1000, random_state=RS),
    {"class_weight": [{0: 0.05, 1: 0.95}, {0: 0.1, 1: 0.9}, {0: 0.2, 1: 0.8}, {0: 0.5, 1: 0.5}]},
    scoring="f1", cv=5)
grid.fit(X_train, y_train)
bobot_hr = grid.best_params_["class_weight"]
print("bobot terbaik dari grid search:", bobot_hr)

In [ ]:
hasil_hr = bandingkan(hr, model_lr, bobot=bobot_hr)
hasil_hr.round(4)

In [ ]:
grafik(hasil_hr, "Prediksi pergantian pekerjaan: logistic regression")

## 7. Perbandingan antar skenario

In [ ]:
ringkas = pd.DataFrame({
    "credit": hasil_credit[f"F{BETA}"],
    "churn": hasil_churn[f"F{BETA}"],
    "tumor": hasil_cancer[f"F{BETA}"],
    "HR": hasil_hr[f"F{BETA}"],
}).round(3)
print(f"F{BETA} tiap cara penanganan di tiap skenario:")
ringkas

In [ ]:
print("cara dengan F-beta tertinggi di tiap skenario:")
print(ringkas.idxmax().to_string())

## Ringkasan

- Pada data yang timpang, akurasi bisa tinggi walaupun model gagal menemukan kelas minoritas. Pakai precision, recall, F-beta, dan AUC.
- Class weight, SMOTE, dan undersampling umumnya menaikkan recall dengan mengorbankan precision. Seberapa besar pengorbanan yang bisa diterima bergantung pada biaya kesalahan di masalah aslinya.
- Resampling hanya boleh diterapkan pada data latih. Data test dibiarkan sesuai distribusi aslinya.
- Untuk data yang sangat timpang, class weight sering lebih aman daripada oversampling. Untuk data yang tidak terlalu timpang dan polanya jelas, penanganan khusus mungkin tidak diperlukan.